# 01 — Pagantes ativos no corte (30/09/2026)

**Pergunta:** quantos assinantes pagantes ativos a Vitaliza tinha no corte?

**Arquivos:** `dados/brutos/assinaturas.csv` e `dados/brutos/usuarios.csv` (dicionário em `docs/`).

**Regras:** contas de teste (e-mail `@vitaliza.com.br`) não contam; mostrar as linhas depois de cada filtro; nada em `dados/brutos/` é alterado.

**Pronto quando:** a soma por plano e por canal bate com o total.

**Referência (enunciado):** 14.753 pagantes; anual 57% / mensal 43%; App Store 36,6%, Google Play 33,6%, cartão 20,3%, Pix 9,5%.

In [1]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
assinaturas = pd.read_csv(BRUTOS / 'assinaturas.csv', dtype=str)
usuarios = pd.read_csv(BRUTOS / 'usuarios.csv', dtype=str)
print(f'assinaturas: {len(assinaturas):,} linhas | usuarios: {len(usuarios):,} linhas')

assinaturas: 32,643 linhas | usuarios: 64,948 linhas


## Qualidade das chaves

`assinaturas.csv` é um retrato de 01/10/2026 (dicionário, seção 03), então `status_atual` já reflete a situação no corte. Antes de filtrar, confirmamos que o cruzamento pelo `user_id` não perde nem duplica linhas.

In [2]:
assert usuarios['user_id'].is_unique, 'user_id duplicado em usuarios'
assert assinaturas['user_id'].isin(usuarios['user_id']).all(), 'assinatura sem conta em usuarios'
print('user_id único em usuarios e todas as assinaturas têm conta.')
for col in ['status_atual', 'plano_atual', 'canal_cobranca']:
    print(col, assinaturas[col].value_counts().to_dict())

user_id único em usuarios e todas as assinaturas têm conta.
status_atual {'expirada': 17758, 'ativa': 14813, 'reembolsada': 72}
plano_atual {'mensal': 19556, 'anual': 13027, 'cortesia': 60}
canal_cobranca {'app_store': 11170, 'google_play': 10948, 'web_cartao': 7087, 'web_pix': 3378, 'interno': 60}


## Contas de teste

O e-mail é digitado pela pessoa e não é validado (dicionário, seção 02): há e-mails com maiúsculas e espaços. Normalizamos antes de extrair o domínio.

In [3]:
email_norm = usuarios['email'].str.strip().str.lower()
usuarios['conta_teste'] = email_norm.str.endswith('@vitaliza.com.br')
print(f"E-mails com maiúscula ou espaço: {(usuarios['email'] != email_norm).sum():,}")
print(f"Contas de teste em usuarios: {usuarios['conta_teste'].sum():,}")

base = assinaturas.merge(usuarios[['user_id', 'email', 'conta_teste']], on='user_id', how='left', validate='many_to_one')
assert len(base) == len(assinaturas)

E-mails com maiúscula ou espaço: 959
Contas de teste em usuarios: 60


## Filtros, passo a passo

In [4]:
passos = [('Todas as assinaturas', base)]
ativas = base[base['status_atual'] == 'ativa']
passos.append(("status_atual == 'ativa'", ativas))
pagantes = ativas[~ativas['conta_teste']]
passos.append(('sem contas de teste (@vitaliza.com.br)', pagantes))

filtros = pd.DataFrame({
    'filtro': [p for p, _ in passos],
    'linhas': [len(df) for _, df in passos],
})
filtros['removidas'] = (-filtros['linhas'].diff()).fillna(0).astype(int)
filtros

,filtro,linhas,removidas
0,Todas as assinaturas,32643,0
1,status_atual == 'ativa',14813,17830
2,sem contas de teste (@vitaliza.com.br),14753,60


In [5]:
# A armadilha: as 60 contas de teste são justamente as do plano 'cortesia', canal 'interno'
removidas = ativas[ativas['conta_teste']]
removidas.groupby(['plano_atual', 'canal_cobranca']).size()

plano_atual  canal_cobranca
cortesia     interno           60
dtype: int64

## Resultado e conferência

In [6]:
total = len(pagantes)
por_plano = pagantes['plano_atual'].value_counts().to_frame('assinantes')
por_plano['%'] = (por_plano['assinantes'] / total * 100).round(1)
por_canal = pagantes['canal_cobranca'].value_counts().to_frame('assinantes')
por_canal['%'] = (por_canal['assinantes'] / total * 100).round(1)

assert por_plano['assinantes'].sum() == total, 'soma por plano não bate'
assert por_canal['assinantes'].sum() == total, 'soma por canal não bate'
assert total == 14_753, f'esperado 14.753 (enunciado), obtido {total:,}'

print(f'Pagantes ativos em 30/09/2026: {total:,}'.replace(',', '.'))
display(por_plano, por_canal)

Pagantes ativos em 30/09/2026: 14.753


,assinantes,%
plano_atual,,
anual,8478,57.5
mensal,6275,42.5


,assinantes,%
canal_cobranca,,
app_store,5395,36.6
google_play,4964,33.6
web_cartao,2994,20.3
web_pix,1400,9.5


## Checagem manual

Três contas do resultado e uma conta de teste removida, com as linhas originais dos dois arquivos. A semente fixa garante que o grupo veja as mesmas contas.

In [7]:
amostra = pagantes['user_id'].sample(3, random_state=42).tolist() + removidas['user_id'].head(1).tolist()
cols_u = ['user_id', 'email', 'data_cadastro', 'canal_aquisicao']
cols_a = ['user_id', 'plano_atual', 'canal_cobranca', 'inicio', 'status_atual', 'renovacao_automatica', 'data_fim']
for uid in amostra:
    print('=' * 80)
    display(usuarios.loc[usuarios['user_id'] == uid, cols_u], assinaturas.loc[assinaturas['user_id'] == uid, cols_a])

,user_id,email,data_cadastro,canal_aquisicao
26017,u_ceb8d4d4,beatriz.campos71@gmail.com,2026-05-08 07:39:49,meta_ads


,user_id,plano_atual,canal_cobranca,inicio,status_atual,renovacao_automatica,data_fim
28011,u_ceb8d4d4,mensal,app_store,2026-05-15,ativa,sim,NaN


,user_id,email,data_cadastro,canal_aquisicao
55316,u_690b4dd3,marcelo_freitas@yahoo.com.br,2021-10-06 12:00:00,influenciador


,user_id,plano_atual,canal_cobranca,inicio,status_atual,renovacao_automatica,data_fim
2864,u_690b4dd3,mensal,google_play,2021-10-13,ativa,sim,NaN


,user_id,email,data_cadastro,canal_aquisicao
25170,u_b7c2766b,olivia.pinto91@uol.com.br,2026-05-15 17:45:42,meta_ads


,user_id,plano_atual,canal_cobranca,inicio,status_atual,renovacao_automatica,data_fim
28228,u_b7c2766b,anual,google_play,2026-05-22,ativa,sim,NaN


,user_id,email,data_cadastro,canal_aquisicao
26407,u_a044e148,qa.0@vitaliza.com.br,2025-06-04 12:19:37,interno


,user_id,plano_atual,canal_cobranca,inicio,status_atual,renovacao_automatica,data_fim
32583,u_a044e148,cortesia,interno,2025-01-10,ativa,nao,NaN


Conferência independente: um pagante ativo deve ter cobrança aprovada recente em `pagamentos.csv` (mensal no último mês, anual nos últimos 12); a conta de teste não deve ter nenhuma.

In [8]:
pagamentos = pd.read_csv(BRUTOS / 'pagamentos.csv', dtype=str)
for uid in amostra:
    p = pagamentos[pagamentos['user_id'] == uid].sort_values('data_hora')
    print(uid, f'{len(p)} cobranças | última:', p.tail(1)[['data_hora', 'valor', 'metodo', 'status']].values.tolist())

u_ceb8d4d4 5 cobranças | última: [['2026-09-13T00:56:46Z', '39.90', 'app_store', 'aprovado']]
u_690b4dd3 28 cobranças | última: [['2026-09-17T12:26:44Z', '39.90', 'google_play', 'aprovado']]
u_b7c2766b 1 cobranças | última: [['2026-05-22T18:36:52Z', '299.90', 'google_play', 'aprovado']]
u_a044e148 0 cobranças | última: []


## Conclusão

- **14.753 pagantes ativos** em 30/09/2026, igual ao enunciado.
- A soma por plano e por canal bate com o total, e os percentuais batem com o enunciado.
- **Armadilha:** contar `status_atual == 'ativa'` sem cruzar com `usuarios.csv` dá 14.813, porque inclui as 60 contas de teste.
- Este número vale para o corte. Para contar ativos em datas passadas (churn mensal), `assinaturas.csv` não serve: é preciso reconstruir a partir de `eventos_assinatura.csv`.